## **Aim**
To implement a program that extracts and analyzes recently accessed files from a simulated system activity log.

## **Algorithm**
**Step 1:** Import `json`, `datetime`, `collections`, and `os` libraries.

**Step 2:** Create a simulated system activity log (JSON format) containing file access events with timestamps, file paths, process names, and user info.

**Step 3:** Define a function `parse_activity_log(log_file)` to load and parse the JSON log.

**Step 4:** Filter events by time range (e.g., last 24 hours) and event type (file read, write, execute).

**Step 5:** Group events by file path and count access frequency per file.

**Step 6:** Identify anomalies: files accessed by unusual processes, accesses at odd hours, high-frequency access to sensitive files.

**Step 7:** Generate a report of recently accessed files sorted by recency and frequency.

In [1]:
import json
import os
from datetime import datetime, timedelta
from collections import defaultdict, Counter

def create_sample_activity_log(log_file):
    now = datetime.now()
    events = []
    
    # Normal user activity
    base_time = now - timedelta(hours=24)
    for i in range(50):
        timestamp = base_time + timedelta(minutes=i*20)
        events.append({
            "timestamp": timestamp.isoformat(),
            "event_type": "FILE_READ",
            "file_path": f"/home/user/documents/report_{i%5}.docx",
            "process": "WINWORD.EXE",
            "user": "user",
            "pid": 1234 + i%10
        })
    
    for i in range(30):
        timestamp = base_time + timedelta(minutes=i*30)
        events.append({
            "timestamp": timestamp.isoformat(),
            "event_type": "FILE_WRITE",
            "file_path": f"/home/user/downloads/data_{i%3}.csv",
            "process": "EXCEL.EXE",
            "user": "user",
            "pid": 2345 + i%5
        })
    
    # Suspicious activity - unusual process accessing sensitive files
    suspicious_files = [
        "/etc/passwd", "/etc/shadow", "/home/user/.ssh/id_rsa",
        "/home/user/.bash_history", "/var/log/auth.log",
        "C:\\Windows\\System32\\config\\SAM",
        "C:\\Users\\Admin\\Documents\\passwords.txt"
    ]
    
    for i, fpath in enumerate(suspicious_files):
        timestamp = now - timedelta(hours=2, minutes=i*15)
        events.append({
            "timestamp": timestamp.isoformat(),
            "event_type": "FILE_READ",
            "file_path": fpath,
            "process": "powershell.exe" if i % 2 == 0 else "cmd.exe",
            "user": "user",
            "pid": 9999 + i
        })
    
    # High frequency access to one file
    for i in range(100):
        timestamp = now - timedelta(minutes=60) + timedelta(seconds=i*30)
        events.append({
            "timestamp": timestamp.isoformat(),
            "event_type": "FILE_READ",
            "file_path": "/home/user/secret_keys/api_key.txt",
            "process": "python3",
            "user": "user",
            "pid": 5555
        })
    
    # Night-time access
    night_time = now.replace(hour=3, minute=15, second=0, microsecond=0)
    for i in range(5):
        events.append({
            "timestamp": (night_time + timedelta(minutes=i*10)).isoformat(),
            "event_type": "FILE_WRITE",
            "file_path": f"/tmp/staged_data_{i}.bin",
            "process": "unknown_process",
            "user": "user",
            "pid": 7777
        })
    
    with open(log_file, "w") as f:
        json.dump(events, f, indent=2)

def analyze_activity_log(log_file, hours_back=24):
    with open(log_file, "r") as f:
        events = json.load(f)
    
    cutoff = datetime.now() - timedelta(hours=hours_back)
    
    # Filter recent events
    recent_events = []
    for e in events:
        ts = datetime.fromisoformat(e["timestamp"])
        if ts >= cutoff:
            recent_events.append(e)
    
    # Group by file
    file_access = defaultdict(list)
    for e in recent_events:
        file_access[e["file_path"]].append(e)
    
    # Process stats
    process_counter = Counter()
    user_counter = Counter()
    hourly_counter = Counter()
    
    for e in recent_events:
        process_counter[e["process"]] += 1
        user_counter[e["user"]] += 1
        ts = datetime.fromisoformat(e["timestamp"])
        hourly_counter[ts.hour] += 1
    
    # Find anomalies
    anomalies = []
    
    # High frequency files
    for fpath, accesses in file_access.items():
        if len(accesses) > 20:
            anomalies.append({
                "type": "HIGH_FREQUENCY",
                "file": fpath,
                "count": len(accesses),
                "processes": list(set(a["process"] for a in accesses)),
                "severity": "HIGH"
            })
    
    # Sensitive files accessed by unusual processes
    sensitive_paths = ["/etc/passwd", "/etc/shadow", "/home/user/.ssh/id_rsa", 
                       "/home/user/.bash_history", "/var/log/auth.log",
                       "C:\\Windows\\System32\\config\\SAM"]
    for fpath, accesses in file_access.items():
        for sens in sensitive_paths:
            if sens in fpath:
                procs = set(a["process"] for a in accesses)
                if any(p in ["powershell.exe", "cmd.exe", "python3", "unknown_process"] for p in procs):
                    anomalies.append({
                        "type": "SENSITIVE_FILE_ACCESS",
                        "file": fpath,
                        "processes": list(procs),
                        "count": len(accesses),
                        "severity": "CRITICAL"
                    })
    
    # Night time access (11PM - 5AM)
    for e in recent_events:
        ts = datetime.fromisoformat(e["timestamp"])
        if 23 <= ts.hour or ts.hour <= 5:
            anomalies.append({
                "type": "OFF_HOURS_ACCESS",
                "file": e["file_path"],
                "time": e["timestamp"],
                "process": e["process"],
                "severity": "MEDIUM"
            })
    
    return {
        "total_events": len(recent_events),
        "unique_files": len(file_access),
        "file_access": file_access,
        "process_stats": dict(process_counter.most_common()),
        "user_stats": dict(user_counter),
        "hourly_stats": dict(sorted(hourly_counter.items())),
        "anomalies": anomalies
    }

def main():
    log_file = "system_activity.json"
    create_sample_activity_log(log_file)
    
    print("Analyzing system activity log...")
    results = analyze_activity_log(log_file, hours_back=24)
    
    print(f"\n{'='*60}")
    print(f"RECENT FILE ACCESS SUMMARY (Last 24 Hours)")
    print(f"{'='*60}")
    print(f"Total Events: {results['total_events']}")
    print(f"Unique Files Accessed: {results['unique_files']}")
    
    print(f"\n--- Top Processes by File Access ---")
    for proc, count in list(results['process_stats'].items())[:10]:
        print(f"  {proc:<25} {count} accesses")
    
    print(f"\n--- Hourly Activity Distribution ---")
    for hour, count in sorted(results['hourly_stats'].items()):
        bar = "#" * (count // 5)
        print(f"  {hour:02d}:00  {count:3d} {bar}")
    
    print(f"\n--- ANOMALIES DETECTED ---")
    if not results['anomalies']:
        print("  No anomalies detected.")
    else:
        for i, a in enumerate(results['anomalies'], 1):
            print(f"\n  {i}. [{a['severity']}] {a['type']}")
            print(f"     File: {a['file']}")
            if 'count' in a:
                print(f"     Access Count: {a['count']}")
            if 'processes' in a:
                print(f"     Processes: {', '.join(a['processes'])}")
            if 'time' in a:
                print(f"     Time: {a['time']}")
            if 'process' in a:
                print(f"     Process: {a['process']}")
    
    # Top recently accessed files
    print(f"\n--- TOP 15 RECENTLY ACCESSED FILES ---")
    file_counts = [(f, len(accesses), max(datetime.fromisoformat(a["timestamp"]) for a in accesses)) 
                   for f, accesses in results['file_access'].items()]
    file_counts.sort(key=lambda x: x[2], reverse=True)
    
    for fpath, count, last_access in file_counts[:15]:
        print(f"  {last_access.strftime('%H:%M:%S')}  {count:3d}x  {fpath}")

if __name__ == "__main__":
    main()

Analyzing system activity log...

RECENT FILE ACCESS SUMMARY (Last 24 Hours)
Total Events: 190
Unique Files Accessed: 21

--- Top Processes by File Access ---
  python3                   100 accesses
  WINWORD.EXE               49 accesses
  EXCEL.EXE                 29 accesses
  unknown_process           5 accesses
  powershell.exe            4 accesses
  cmd.exe                   3 accesses

--- Hourly Activity Distribution ---
  00:00    3 
  01:00    2 
  03:00    5 #
  05:00    2 
  06:00    4 
  07:00    1 
  08:00  100 ####################
  09:00    3 
  10:00    5 #
  11:00    5 #
  12:00    5 #
  13:00    5 #
  14:00    5 #
  15:00    5 #
  16:00    5 #
  17:00    5 #
  18:00    5 #
  19:00    5 #
  20:00    5 #
  21:00    5 #
  22:00    5 #
  23:00    5 #

--- ANOMALIES DETECTED ---

  1. [HIGH] HIGH_FREQUENCY
     File: /home/user/secret_keys/api_key.txt
     Access Count: 100
     Processes: python3

  2. [CRITICAL] SENSITIVE_FILE_ACCESS
     File: /etc/passwd
     Access

## **Result**
This the program successfully extracts and analyzes recently accessed files from a simulated system activity log.